# TrueVoice — Gemma 4 E4B Evaluation (Linear Probe)

**Goal**: Restore the trained Linear Probe checkpoint and
evaluate on ASVspoof 2021 LA eval dataset (EER) + save model

---

## Execution Order
Cell 1 : Install packages

Cell 2 : Mount Drive + extract audio archives

Cell 3 : Define preprocessing functions

Cell 4 : Load model & processor

Cell 5 : Define AudioDeepfakeClassifier

Cell 6 : Restore checkpoint (checkpoint-4761)

Cell 7 : Load dataset & prepare collator

Cell 8 : Define compute_metrics

Cell 9 : Evaluate on ASVspoof 2021 LA eval (EER + classification_report)

Cell 10: Save model

---

## Checklist
- [ ] Colab Pro — select A100 GPU runtime
- [ ] HuggingFace token ready (`google/gemma-4-e4b-it`)
- [ ] `checkpoints/checkpoint-4761` saved in Google Drive
- [ ] `test_2021.json` saved in Google Drive
- [ ] `ASVspoof2021_LA_eval.tar.gz`, `keys.tar.gz` saved in Drive
- [ ] ⛔ Do NOT run any LoRA cell — it corrupts base model weights


## Cell 1: Install Packages

> Estimated time: ~3-5 minutes

In [ ]:
%%capture
# Audio processing
!pip install librosa soundfile torchaudio

# Training utilities
!pip install transformers datasets accelerate scikit-learn scipy
!pip install git+https://github.com/huggingface/transformers.git

print('Packages installed successfully')

## Cell 2: Mount Drive + Extract Audio Archives

In [ ]:
from google.colab import drive
import os
import glob

drive.mount('/content/drive', force_remount=True)

# Path constants
DRIVE_BASE    = '/content/drive/MyDrive/2026truevoice'
DRIVE_DATASET = f'{DRIVE_BASE}/dataset'
LOCAL_BASE    = '/content/asvspoof'
CKPT_DIR      = f'{DRIVE_BASE}/checkpoints'

os.makedirs(LOCAL_BASE, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)


def fast_copy_and_extract(archive_name, extract_path):
    """Copy archive from Drive to local SSD, then extract.
    Skips if the target directory already exists and is non-empty.
    """
    if os.path.exists(extract_path) and len(os.listdir(extract_path)) > 0:
        print(f'  Already exists, skipping: {extract_path}')
        return

    src            = f'{DRIVE_DATASET}/{archive_name}'
    temp_local_zip = f'/content/{archive_name}'

    print(f'  Copying to local SSD: {archive_name}')
    os.system(f'cp "{src}" "{temp_local_zip}"')

    print(f'  Extracting: {archive_name}')
    if archive_name.endswith('.zip'):
        os.system(f'unzip -q "{temp_local_zip}" -d "{LOCAL_BASE}/"')
    else:
        os.system(f'tar -xzf "{temp_local_zip}" -C "{LOCAL_BASE}/"')

    # Remove local archive to free up disk space
    os.remove(temp_local_zip)
    print(f'  Done: {archive_name}')


print('=== Extracting archives ===')

# Small archives first
fast_copy_and_extract('keys.tar.gz',                 f'{LOCAL_BASE}/keys')
fast_copy_and_extract('ASVspoof2021_LA_eval.tar.gz', f'{LOCAL_BASE}/ASVspoof2021_LA_eval')

# ASVspoof 2019 (large — most time-consuming)
ASV19_CHECK = f'{LOCAL_BASE}/LA/LA/ASVspoof2019_LA_train'
if not os.path.exists(ASV19_CHECK):
    print('Extracting ASVspoof2019 (large file)...')
    fast_copy_and_extract('ASVspoof2019.zip', f'{LOCAL_BASE}/LA')
else:
    print('  ASVspoof2019 already exists, skipping')

# Verify
flac_2019 = glob.glob('/content/asvspoof/LA/LA/ASVspoof2019_LA_train/flac/*.flac')
print(f'\nReady: {len(flac_2019)} 2019 train flac files found')

## Cell 3: Define Preprocessing Functions

In [ ]:
import os
import json
import random
import torch
import torchaudio
import librosa
from dataclasses import dataclass
from typing import List, Dict
from datasets import Dataset

# Path setup
BASE_2019       = LOCAL_BASE
TRAIN_PROTO     = f'{BASE_2019}/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
VAL_PROTO       = f'{BASE_2019}/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt'
TRAIN_AUDIO_DIR = f'{BASE_2019}/ASVspoof2019_LA_train/flac'
VAL_AUDIO_DIR   = f'{BASE_2019}/ASVspoof2019_LA_dev/flac'
BASE_2021_AUDIO = f'{LOCAL_BASE}/ASVspoof2021_LA_eval/flac'
BASE_2021_KEY   = f'{LOCAL_BASE}/keys/LA/CM/trial_metadata.txt'


def codec_simulate(audio_np, sr=16000):
    """Simulate phone codec: downsample to 8kHz then upsample back to 16kHz.
    Removes the studio-quality domain fingerprint of ASVspoof 2019 audio.
    """
    waveform = torch.from_numpy(audio_np).unsqueeze(0).float()
    down = torchaudio.functional.resample(waveform, orig_freq=sr, new_freq=8000)
    up   = torchaudio.functional.resample(down,     orig_freq=8000, new_freq=sr)
    return up.squeeze(0).numpy()


def load_audio(path, sr=16000, max_sec=29.0, apply_codec=False):
    """Load audio, clip to max_sec, and optionally apply codec simulation."""
    audio, _ = librosa.load(path, sr=sr, mono=True)
    max_samples = int(max_sec * sr)
    if len(audio) > max_samples:
        audio = audio[:max_samples]
    if apply_codec:
        audio = codec_simulate(audio, sr=sr)
    return audio


def make_clf_dataset(samples, max_samples=None):
    """Store only file paths and labels — audio is loaded per batch by the collator."""
    if max_samples:
        samples = random.sample(samples, min(max_samples, len(samples)))
    return Dataset.from_dict({
        'audio_path': [s['audio_path'] for s in samples],
        'labels':     [0 if s['label'] == 'real' else 1 for s in samples],
    })


@dataclass
class AudioDataCollator:
    """On-the-fly audio loader: loads audio per batch, converts to mel spectrogram, pads.

    Gemma4 processor returns input_features of shape (time, mel=128).
    Padding is applied along the time axis (dim=0).

    Args:
        processor   : Gemma4 AutoProcessor
        apply_codec : True  for train/val (codec simulation on 2019 studio audio)
                      False for test  (2021 LA already has real codec effects)
    """
    processor: object
    apply_codec: bool = True

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        batch_features, batch_labels = [], []

        for item in features:
            try:
                audio  = load_audio(item['audio_path'], apply_codec=self.apply_codec)
                inputs = self.processor(
                    text='<audio>',
                    audio=audio,
                    sampling_rate=16000,
                    return_tensors='pt',
                )
                if 'input_features' not in inputs:
                    continue
                batch_features.append(inputs['input_features'][0])
                batch_labels.append(item['labels'])
            except Exception:
                continue  # skip corrupted or missing files silently

        if not batch_features:
            raise RuntimeError('No valid audio samples in batch.')

        # Pad along time axis (dim=0) — mel axis (dim=1) is fixed at 128
        max_len = max(f.shape[0] for f in batch_features)
        padded  = torch.stack([
            torch.nn.functional.pad(f, (0, 0, 0, max_len - f.shape[0]))
            for f in batch_features
        ])

        return {
            'input_features': padded.float(),
            'labels': torch.tensor(batch_labels, dtype=torch.long),
        }

print('Preprocessing functions defined')

## Cell 4: Load Model & Processor

> **Important**: `dtype=torch.float32` is required.
> bfloat16 causes NaN in the audio_tower forward pass.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

MODEL_ID = 'google/gemma-4-e4b-it'
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device     : {DEVICE}')
print(f'GPU memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
print('Processor loaded')

# dtype=torch.float32 is mandatory — bfloat16 causes NaN in audio_tower
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    dtype=torch.float32,
    device_map='auto',
)
print('Base model loaded')

audio_tower = base_model.model.audio_tower
hidden_size = audio_tower.output_proj.out_features  # 1536
print(f'audio_tower hidden_size: {hidden_size}')

## Cell 5: Define AudioDeepfakeClassifier

In [ ]:
import torch.nn as nn
from transformers.modeling_outputs import SequenceClassifierOutput


class AudioDeepfakeClassifier(nn.Module):
    """Binary classifier using Gemma4's audio_tower as a frozen feature extractor.

    Architecture:
        input_features  (mel spectrogram, shape: batch x time x 128)
            -> audio_tower  (frozen, outputs 1536-dim hidden states)
            -> mean pooling over time axis  -> (batch, 1536)
            -> Linear(1536->256) -> GELU -> Dropout(0.3)
            -> Linear(256->2)   [real=0, fake=1]
    """

    def __init__(self, audio_tower, hidden_size: int, num_labels: int = 2, dropout_p: float = 0.3):
        super().__init__()
        self.audio_tower = audio_tower
        self.num_labels  = num_labels
        self.classifier  = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_features: torch.Tensor, labels: torch.Tensor = None):
        # Cast to float32 explicitly to prevent dtype mismatch
        tower_out = self.audio_tower(input_features=input_features.float())

        if hasattr(tower_out, 'last_hidden_state'):
            hidden = tower_out.last_hidden_state
        else:
            hidden = tower_out[0]

        pooled = hidden.mean(dim=1)     # mean pooling -> (batch, hidden_size)
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return SequenceClassifierOutput(loss=loss, logits=logits)


print('AudioDeepfakeClassifier defined')

## Cell 6: Restore Checkpoint

In [ ]:
import torch, os
from safetensors.torch import load_file

# Freeze the entire audio_tower
for param in base_model.model.audio_tower.parameters():
    param.requires_grad = False

# Do NOT call .to(DEVICE) on clf_model — audio_tower is already on GPU via device_map='auto'
clf_model = AudioDeepfakeClassifier(
    audio_tower=base_model.model.audio_tower,
    hidden_size=hidden_size,
)
# Move only the classifier head to GPU
clf_model.classifier = clf_model.classifier.to(DEVICE).to(torch.float32)

# Restore classifier head weights from checkpoint
CKPT = f'{DRIVE_BASE}/checkpoints/checkpoint-4761'
state_dict = load_file(os.path.join(CKPT, 'model.safetensors'), device='cpu')
classifier_state = {
    k.replace('classifier.', ''): v
    for k, v in state_dict.items()
    if k.startswith('classifier.')
}
clf_model.classifier.load_state_dict(classifier_state)
print('Checkpoint restored successfully')

## Cell 7: Load Dataset & Prepare Collator

In [ ]:
import json

TRAIN_JSON = f'{DRIVE_BASE}/train.json'
VAL_JSON   = f'{DRIVE_BASE}/val.json'
TEST_JSON  = f'{DRIVE_BASE}/test_2021.json'

with open(TRAIN_JSON) as f: train_samples = json.load(f)
with open(VAL_JSON)   as f: val_samples   = json.load(f)
with open(TEST_JSON)  as f: test_samples  = json.load(f)

print(f'train : {len(train_samples):,} samples')
print(f'val   : {len(val_samples):,} samples')
print(f'test  : {len(test_samples):,} samples')

# apply_codec=False for test — 2021 LA already contains real codec effects
train_collator = AudioDataCollator(processor=processor, apply_codec=True)
test_collator  = AudioDataCollator(processor=processor, apply_codec=False)

print('Dataset and collators ready')

## Cell 8: Define compute_metrics

In [ ]:
import numpy as np
from sklearn.metrics import f1_score


def compute_metrics(eval_pred):
    """Compute F1-macro and accuracy. Used for checkpoint selection during training."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1  = f1_score(labels, preds, average='macro', zero_division=0)
    acc = (preds == labels).mean()
    return {'f1_macro': f1, 'accuracy': acc}


print('compute_metrics defined')

## Cell 9: Evaluate on ASVspoof 2021 LA eval (Full — 148,176 samples)

> Estimated time: ~3 hours on A100

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import classification_report, roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from tqdm import tqdm


def evaluate_2021(model, processor, samples, device='cuda', batch_size=128):
    """Evaluate on the full ASVspoof 2021 LA eval set.
    Computes EER (Equal Error Rate) and a per-class classification report.

    Args:
        model      : trained AudioDeepfakeClassifier
        processor  : Gemma4 AutoProcessor
        samples    : list of {audio_path, label} dicts
        device     : 'cuda'
        batch_size : samples per batch (128 recommended for A100)

    Returns:
        dict with eer, y_true, y_pred, y_score
    """
    model.eval()
    y_true, y_pred, y_score = [], [], []
    failed = 0

    for i in tqdm(range(0, len(samples), batch_size), desc='Evaluating'):
        batch = samples[i : i + batch_size]
        loaded_features, loaded_samples = [], []

        for sample in batch:
            try:
                # No codec simulation — 2021 eval already has real codec effects
                audio  = load_audio(sample['audio_path'], apply_codec=False)
                inputs = processor(
                    text='<audio>', audio=audio,
                    sampling_rate=16000, return_tensors='pt',
                )
                if 'input_features' not in inputs:
                    continue
                loaded_features.append(inputs['input_features'][0])
                loaded_samples.append(sample)
            except Exception:
                failed += 1
                continue

        if not loaded_features:
            continue

        # Pad along time axis (dim=0)
        max_len = max(f.shape[0] for f in loaded_features)
        padded  = torch.stack([
            torch.nn.functional.pad(f, (0, 0, 0, max_len - f.shape[0]))
            for f in loaded_features
        ]).to(device)

        with torch.no_grad():
            output = model(input_features=padded)
            probs  = torch.softmax(output.logits, dim=-1)
            preds  = output.logits.argmax(-1).cpu().tolist()
            scores = probs[:, 1].cpu().tolist()   # fake class probability

        for j, sample in enumerate(loaded_samples):
            y_true.append(0 if sample['label'] == 'real' else 1)
            y_pred.append(preds[j])
            y_score.append(scores[j])

    # Compute EER via ROC curve interpolation
    fpr, tpr, _ = roc_curve(y_true, y_score, pos_label=1)
    eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)

    print(f'\n{"="*50}')
    print('ASVspoof 2021 LA eval Results')
    print(f'{"="*50}')
    print(f'Evaluated : {len(y_true):,} samples  (failed: {failed})')
    print(f'\nEER : {eer*100:.2f}%  (lower is better, target: <5%)')
    print(f'\n{classification_report(y_true, y_pred, labels=[0,1], target_names=["real","fake"])}')

    return {'eer': eer, 'y_true': y_true, 'y_pred': y_pred, 'y_score': y_score}


print('=== Starting full 2021 LA eval evaluation ===')
eval_results = evaluate_2021(
    model=clf_model,
    processor=processor,
    samples=test_samples,
    device=DEVICE,
    batch_size=128,
)

## Cell 10: Save Model

In [ ]:
import torch, json, datetime, os

SAVE_DIR = f'{DRIVE_BASE}/saved_model'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save classification head weights only (1.5MB)
# The audio_tower is NOT saved — it is reloaded from HuggingFace at inference time
torch.save(clf_model.classifier.state_dict(), f'{SAVE_DIR}/classifier_head.pt')
print(f'Classifier head saved: {SAVE_DIR}/classifier_head.pt')

# Save training metadata
meta = {
    'model_id'        : MODEL_ID,
    'hidden_size'     : hidden_size,
    'method'          : 'linear_probe',
    'eer_2021_full'   : eval_results['eer'],
    'train_samples'   : len(train_samples),
    'val_samples'     : len(val_samples),
    'val_f1_epoch1'   : 0.930,
    'val_f1_epoch2'   : 0.967,
    'val_f1_epoch3'   : 0.973,
    'saved_at'        : datetime.datetime.now().isoformat(),
}
with open(f'{SAVE_DIR}/metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Metadata saved')
print(f'\nFinal EER : {eval_results["eer"]*100:.2f}%')
print(f'Save path : {SAVE_DIR}')